In [0]:
%pip install -U "typing_extensions>=4.14,<5" databricks-vectorsearch databricks-ai-search databricks-langchain

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
CATALOG = "cs4603"
SCHEMA = "pa4"
VOLUME = "documents"

PDF_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/annual_report.pdf"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.pa4_analyst_chunks"

VECTOR_SEARCH_ENDPOINT = "pa4-vs-endpoint"
VECTOR_SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.pa4_analyst_index"
EMBEDDINGS_ENDPOINT = "databricks-gte-large-en"

print("PDF path:", PDF_PATH)
print("Source table:", SOURCE_TABLE)
print("Vector Search endpoint:", VECTOR_SEARCH_ENDPOINT)
print("Vector Search index:", VECTOR_SEARCH_INDEX)
print("Embedding endpoint:", EMBEDDINGS_ENDPOINT)

PDF path: /Volumes/cs4603/pa4/documents/annual_report.pdf
Source table: cs4603.pa4.pa4_analyst_chunks
Vector Search endpoint: pa4-vs-endpoint
Vector Search index: cs4603.pa4.pa4_analyst_index
Embedding endpoint: databricks-gte-large-en


In [0]:
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}"
)

print(
    "Volume ready:",
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/",
)

Volume ready: /Volumes/cs4603/pa4/documents/


In [0]:
files = dbutils.fs.ls(
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"
)

display(files)

assert any(
    file.name == "annual_report.pdf"
    for file in files
), (
    f"annual_report.pdf was not found in "
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"
)

print("Found:", PDF_PATH)

path,name,size,modificationTime
dbfs:/Volumes/cs4603/pa4/documents/annual_report.pdf,annual_report.pdf,21839,1783791434000


Found: /Volumes/cs4603/pa4/documents/annual_report.pdf


In [0]:
escaped_pdf_path = PDF_PATH.replace("'", "''")

parsed_df = spark.sql(
    f"""
    SELECT
        path,
        ai_parse_document(
            content,
            map('version', '2.0')
        ) AS document
    FROM read_files(
        '{escaped_pdf_path}',
        format => 'binaryFile'
    )
    """
)

display(parsed_df)

path document dbfs:/Volumes/cs4603/pa4/documents/annual_report.pdf {"document":{"elements":[{"bbox":[{"coord":[195,417,996,489],"page_id":0}],"confidence":0.9998,"content":"Meridian Motor Corporation","description":null,"id":0,"type":"title"},{"bbox":[{"coord":[493,519,699,558],"page_id":0}],"confidence":0.9997,"content":"Annual Report","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[251,556,941,600],"page_id":0}],"confidence":1,"content":"For the fiscal year ended March 31, 2023 (FY2023)","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[249,654,944,697],"page_id":0}],"confidence":0.992,"content":"Tokyo Stock Exchange (Prime Market) · Code 7000","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[109,1601,501,1629],"page_id":0}],"confidence":0.9991,"content":"Meridian Motor Corporation — Annual Report FY2023","description":null,"id":4,"type":"page_footer"},{"bbox":[{"coord":[1022,1601,1081,1629],"page_id":0}],"confidence":0.9996,"content":"Page 1","description":null,"id":5,"type":"page_number"},{"bbox":[{"coord":[122,128,415,164],"page_id":1}],"confidence":0.9997,"content":"Table of Contents","description":null,"id":6,"type":"title"},{"bbox":[{"coord":[121,194,650,663],"page_id":1}],"confidence":0.996,"content":" Letter from the President and CEO 3 Financial Highlights — Five-Year Summary 4 Consolidated Statement of Operations 5 About Meridian Motor Corporation 6 Segment Information 7 Regional Performance 8 Consolidated Balance Sheet 9 Consolidated Statements of Cash Flows 10 Research & Development and Capital Investment 11 Risk Factors 12 Outlook and Guidance (FY2024) 13 Notes and Glossary 14 ","description":null,"id":7,"type":"table"},{"bbox":[{"coord":[121,709,872,735],"page_id":1}],"confidence":0.9991,"content":"All figures in this report are fictional and provided solely for CS4603 coursework.","description":null,"id":8,"type":"text"},{"bbox":[{"coord":[112,1605,497,1628],"page_id":1}],"confidence":0.9988,"content":"Meridian Motor Corporation — Annual Report FY2023","description":null,"id":9,"type":"page_footer"},{"bbox":[{"coord":[1024,1605,1078,1628],"page_id":1}],"confidence":0.9996,"content":"Page 2","description":null,"id":10,"type":"page_number"},{"bbox":[{"coord":[122,128,686,165],"page_id":2}],"confidence":0.9999,"content":"Letter from the President and CEO","description":null,"id":11,"type":"section_header"},{"bbox":[{"coord":[121,194,1070,312],"page_id":2}],"confidence":0.9949,"content":"To our shareholders, customers, and employees: Meridian Motor Corporation delivered record results in the fiscal year ended March 31, 2023. Consolidated net revenue rose 16.2% to ¥16,910 billion (¥16.91 trillion), and operating profit increased 24.2% to ¥1,124 billion. Net income attributable to owners of the parent reached ¥1,107 billion, our highest ever.","description":null,"id":12,"type":"text"},{"bbox":[{"coord":[121,331,1070,451],"page_id":2}],"confidence":0.9965,"content":"Demand recovered strongly across all major markets as supply-chain constraints eased. Global vehicle unit sales grew to 4.07 million units, up from 3.68 million a year earlier, led by our electrified line-up in North America and Asia. Our Motorcycle business remained the profitability anchor of the group, with a 14.3% operating margin.","description":null,"id":13,"type":"text"},{"bbox":[{"coord":[121,470,1070,558],"page_id":2}],"confidence":0.9865,"content":"We continued to invest for the long term. Research and development expense rose to ¥880 billion, or 5.2% of net revenue, concentrated on battery-electric platforms, software-defined vehicle architecture, and advanced driver assistance. Capital expenditure was ¥590 billion.","description":null,"id":14,"type":"text"},{"bbox":[{"coord":[121,577,1070,667],"page_id":2}],"confidence":0.9798,"content":"Looking ahead to FY2024, we forecast net revenue of ¥18.20 trillion and operating profit of ¥1.30 trillion, and we have raised the annual dividend to ¥150 per share. On behalf 

In [0]:
prepared_df = spark.sql(
    f"""
    WITH parsed AS (
        SELECT
            path,
            ai_parse_document(
                content,
                map('version', '2.0')
            ) AS document
        FROM read_files(
            '{escaped_pdf_path}',
            format => 'binaryFile'
        )
    )
    SELECT
        path,
        ai_prep_search(document) AS chunks
    FROM parsed
    """
)

prepared_df.printSchema()
display(prepared_df)

root
 |-- path: string (nullable = true)
 |-- chunks: variant (nullable = true)



path chunks dbfs:/Volumes/cs4603/pa4/documents/annual_report.pdf {"document":{"contents":[{"chunk_id":"22ee86c2060f401fab311c83d67013d7_0","chunk_position":0,"chunk_to_embed":"Company: Meridian Motor Corporation\nDocument Type: Annual Report\nFiscal Year End: March 31, 2023\nReport Year: FY2023\nStock Exchange: Tokyo Stock Exchange (Prime Market)\nTicker Code: 7000\nDocument Title: Table of Contents\nTables: Tokyo Stock Exchange (Prime Market) · Code 7000\nContains: table, text\n\nMeridian Motor Corporation's FY2023 Annual Report covering the fiscal year ended March 31, 2023.\n\nTable summary: The table lists sections of Meridian Motor Corporation’s FY2023 report, covering the fiscal year ended March 31 2023.\n\nContent:\n\nAnnual Report\n\nFor the fiscal year ended March 31, 2023 (FY2023)\n\nTokyo Stock Exchange (Prime Market) · Code 7000\n\n Letter from the President and CEO 3 Financial Highlights — Five-Year Summary 4 Consolidated Statement of Operations 5 About Meridian Motor Corporation 6 Segment Information 7 Regional Performance 8 Consolidated Balance Sheet 9 Consolidated Statements of Cash Flows 10 Research & Development and Capital Investment 11 Risk Factors 12 Outlook and Guidance (FY2024) 13 Notes and Glossary 14 \n\n\nRelated questions:\nWhere can I find the Consolidated Balance Sheet? \nWhich section contains the FY2024 Outlook and Guidance? \nWhat part of the report includes Segment Information? \nWhere are the Financial Highlights — Five-Year Summary located? \nIn which section can I read the Letter from the President and CEO?","chunk_to_retrieve":"\nAnnual Report\n\nFor the fiscal year ended March 31, 2023 (FY2023)\n\nTokyo Stock Exchange (Prime Market) · Code 7000\n\n Letter from the President and CEO 3 Financial Highlights — Five-Year Summary 4 Consolidated Statement of Operations 5 About Meridian Motor Corporation 6 Segment Information 7 Regional Performance 8 Consolidated Balance Sheet 9 Consolidated Statements of Cash Flows 10 Research & Development and Capital Investment 11 Risk Factors 12 Outlook and Guidance (FY2024) 13 Notes and Glossary 14 \n","pages":[{"image_uri":"","page_id":0},{"image_uri":"","page_id":1}]},{"chunk_id":"22ee86c2060f401fab311c83d67013d7_1","chunk_position":1,"chunk_to_embed":"Company: Meridian Motor Corporation\nDocument Type: Annual Report\nFiscal Year End: March 31, 2023\nReport Year: FY2023\nStock Exchange: Tokyo Stock Exchange (Prime Market)\nTicker Code: 7000\nDocument Title: Financial Highlights — Five-Year Summary\nSection: Letter from the President and CEO\nTables: over the past five fiscal years. In FY2023, net revenue was ¥16.91 trillion and net income attrib... (¥ billions (except per-share), FY2019, FY2020, FY2021, FY2022, ...)\nContains: table, text\n\nMeridian Motor Corporation's FY2023 Annual Report covering the fiscal year ended March 31, 2023.\n\nTable summary: Financial and operational metrics of Meridian Motor Corp. FY2019‑FY2023, covering revenue, profit, assets, sales, and employees.\n\nContent:\n\nAll figures in this report are fictional and provided solely for CS4603 coursework.\n\nTo our shareholders, customers, and employees: Meridian Motor Corporation delivered record results in the fiscal year ended March 31, 2023. Consolidated net revenue rose 16.2% to ¥16,910 billion (¥16.91 trillion), and operating profit increased 24.2% to ¥1,124 billion. Net income attributable to owners of the parent reached ¥1,107 billion, our highest ever.\n\nDemand recovered strongly across all major markets as supply-chain constraints eased. Global vehicle unit sales grew to 4.07 million units, up from 3.68 million a year earlier, led by our electrified line-up in North America and Asia. Our Motorcycle business remained the profitability anchor of the group, with a 14.3% operating margin.\n\nWe continued to invest for the long term. Research and development expense rose to ¥880 billion, or 5.2% of net revenue, concentrated on battery-electric platforms, software-defined vehicle 

In [0]:
spark.sql(
    f"""
    CREATE OR REPLACE TABLE {SOURCE_TABLE}
    TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )
    AS

    WITH parsed AS (
        SELECT
            path,
            ai_parse_document(
                content,
                map('version', '2.0')
            ) AS document
        FROM read_files(
            '{escaped_pdf_path}',
            format => 'binaryFile'
        )
    ),

    prepared AS (
        SELECT
            path,
            ai_prep_search(document) AS chunks
        FROM parsed
    ),

    exploded AS (
        SELECT
            prepared.path,
            chunk.value AS chunk
        FROM prepared,
        LATERAL variant_explode(prepared.chunks) AS root,
        LATERAL variant_explode(root.value:contents) AS chunk
        WHERE root.key = 'document'
    )

    SELECT
        coalesce(chunk:chunk_id::STRING, uuid()) AS chunk_id,

        chunk:chunk_to_retrieve::STRING
            AS chunk_to_retrieve,

        chunk:chunk_to_embed::STRING
            AS chunk_to_embed,

        regexp_extract(
            path,
            '[^/]+$',
            0
        ) AS source,

        coalesce(
            try_cast(chunk:pages[0].page_id AS INT) + 1,
            try_cast(chunk:metadata.page_number AS INT),
            try_cast(chunk:metadata.page AS INT),
            0
        ) AS page

    FROM exploded

    WHERE
        chunk:chunk_to_retrieve IS NOT NULL
        AND trim(chunk:chunk_to_retrieve::STRING) <> ''
        AND chunk:chunk_to_embed IS NOT NULL
        AND trim(chunk:chunk_to_embed::STRING) <> ''
    """
)

print("Created table:", SOURCE_TABLE)

Created table: cs4603.pa4.pa4_analyst_chunks


In [0]:
chunks_df = spark.table(SOURCE_TABLE)

chunk_count = chunks_df.count()

empty_count = chunks_df.where(
    """
    chunk_to_embed IS NULL
    OR trim(chunk_to_embed) = ''
    """
).count()

print("Total valid chunks:", chunk_count)
print("Empty chunks:", empty_count)

assert chunk_count > 0, (
    "No usable text chunks were produced."
)

assert empty_count == 0, (
    "Some chunks still contain no text."
)

display(
    chunks_df.select(
        "chunk_id",
        "chunk_to_retrieve",
        "chunk_to_embed",
        "source",
        "page",
    )
)

Total valid chunks: 7
Empty chunks: 0


chunk_id chunk_to_retrieve chunk_to_embed source page 4db559e9211140c8b8d4aa2e947bbba3_0 
Annual Report

For the fiscal year ended March 31, 2023 (FY2023)

Tokyo Stock Exchange (Prime Market) · Code 7000

 Letter from the President and CEO 3 Financial Highlights — Five-Year Summary 4 Consolidated Statement of Operations 5 About Meridian Motor Corporation 6 Segment Information 7 Regional Performance 8 Consolidated Balance Sheet 9 Consolidated Statements of Cash Flows 10 Research & Development and Capital Investment 11 Risk Factors 12 Outlook and Guidance (FY2024) 13 Notes and Glossary 14 
 Code: 7000
Company: Meridian Motor Corporation
Document Type: Annual Report
Fiscal Year: FY2023
Stock Exchange: Tokyo Stock Exchange (Prime Market)
Document Title: Table of Contents
Tables: Tokyo Stock Exchange (Prime Market) · Code 7000
Contains: table, text

Meridian Motor Corporation's FY2023 Annual Report covering fiscal year ending March 31, 2023.

Table summary: List of FY2023 annual report sections, covering the fiscal year ending March 31 2023.

Content:

Annual Report

For the fiscal year ended March 31, 2023 (FY2023)

Tokyo Stock Exchange (Prime Market) · Code 7000

 Letter from the President and CEO 3 Financial Highlights — Five-Year Summary 4 Consolidated Statement of Operations 5 About Meridian Motor Corporation 6 Segment Information 7 Regional Performance 8 Consolidated Balance Sheet 9 Consolidated Statements of Cash Flows 10 Research & Development and Capital Investment 11 Risk Factors 12 Outlook and Guidance (FY2024) 13 Notes and Glossary 14 


Related questions:
Where can I find the Consolidated Statement of Operations? 
What section contains the Five-Year Financial Highlights? 
In which part of the report is the Outlook and Guidance for FY2024 located? 
Which section provides Segment Information? 
Where is the Research & Development and Capital Investment information? annual_report.pdf 1 4db559e9211140c8b8d4aa2e947bbba3_1 
All figures in this report are fictional and provided solely for CS4603 coursework.

To our shareholders, customers, and employees: Meridian Motor Corporation delivered record results in the fiscal year ended March 31, 2023. Consolidated net revenue rose 16.2% to ¥16,910 billion (¥16.91 trillion), and operating profit increased 24.2% to ¥1,124 billion. Net income attributable to owners of the parent reached ¥1,107 billion, our highest ever.

Demand recovered strongly across all major markets as supply-chain constraints eased. Global vehicle unit sales grew to 4.07 million units, up from 3.68 million a year earlier, led by our electrified line-up in North America and Asia. Our Motorcycle business remained the profitability anchor of the group, with a 14.3% operating margin.

We continued to invest for the long term. Research and development expense rose to ¥880 billion, or 5.2% of net revenue, concentrated on battery-electric platforms, software-defined vehicle architecture, and advanced driver assistance. Capital expenditure was ¥590 billion.

Looking ahead to FY2024, we forecast net revenue of ¥18.20 trillion and operating profit of ¥1.30 trillion, and we have raised the annual dividend to ¥150 per share. On behalf of the Board, thank you for your continued trust in Meridian Motor Corporation.

The table below summarizes the consolidated performance of Meridian Motor Corporation over the past five fiscal years. In FY2023, net revenue was ¥16.91 trillion and net income attributable to owners was ¥1,107 billion (¥1.11 trillion).

 ¥ billions (except per-share) FY2019 FY2020 FY2021 FY2022 FY2023 Net revenue 11,280 12,110 13,170 14,550 16,910 Operating profit 720 610 880 905 1,124 Net income (owners) 512 455 657 707 1,107 Operating margin 6.4% 5.0% 6.7% 6.2% 6.6% R&D expense 690 705 760 815 880 Capital expenditure 480 455 500 540 590 Total assets 19,900 20,600 22,100 23,700 25,300 Total equity 8,600 8,900 9,800 10,400 11,200 EPS (¥) 296 263 380 409 640 Dividend per share (¥) 100 100 120 135 150 Vehicle unit s

In [0]:
display(
    spark.sql(
        f"""
        WITH parsed AS (
            SELECT
                ai_parse_document(
                    content,
                    map('version', '2.0')
                ) AS document
            FROM read_files(
                '{escaped_pdf_path}',
                format => 'binaryFile'
            )
        ),
        prepared AS (
            SELECT
                ai_prep_search(document) AS chunks
            FROM parsed
        )
        SELECT
            chunk.pos,
            chunk.value,
            schema_of_variant(chunk.value) AS chunk_schema
        FROM prepared,
        LATERAL variant_explode(prepared.chunks) AS chunk
        LIMIT 10
        """
    )
)

pos value chunk_schema 0 {"contents":[{"chunk_id":"18c0eb6d4d8e4e95b35d8ff78966de40_0","chunk_position":0,"chunk_to_embed":"Company: Meridian Motor Corporation\nDocument Type: Annual Report\nFiscal Year: FY2023\nStock Code: 7000\nStock Exchange: Tokyo Stock Exchange (Prime Market)\nDocument Title: Table of Contents\nTables: Tokyo Stock Exchange (Prime Market) · Code 7000\nContains: table, text\n\nMeridian Motor Corporation created this annual report for fiscal year ended March 31, 2023.\n\nTable summary: The table lists the numbered sections of Meridian Motor Corporation’s FY2023 annual report (ended March 31 2023).\n\nContent:\n\nAnnual Report\n\nFor the fiscal year ended March 31, 2023 (FY2023)\n\nTokyo Stock Exchange (Prime Market) · Code 7000\n\n Letter from the President and CEO 3 Financial Highlights — Five-Year Summary 4 Consolidated Statement of Operations 5 About Meridian Motor Corporation 6 Segment Information 7 Regional Performance 8 Consolidated Balance Sheet 9 Consolidated Statements of Cash Flows 10 Research & Development and Capital Investment 11 Risk Factors 12 Outlook and Guidance (FY2024) 13 Notes and Glossary 14 \n\n\nRelated questions:\nWhat section number contains the Consolidated Balance Sheet? \nWhich part of the report discusses Regional Performance? \nWhere can I find the Outlook and Guidance for FY2024? \nIn which section are the Financial Highlights — Five-Year Summary listed? \nWhich section provides Segment Information?","chunk_to_retrieve":"\nAnnual Report\n\nFor the fiscal year ended March 31, 2023 (FY2023)\n\nTokyo Stock Exchange (Prime Market) · Code 7000\n\n Letter from the President and CEO 3 Financial Highlights — Five-Year Summary 4 Consolidated Statement of Operations 5 About Meridian Motor Corporation 6 Segment Information 7 Regional Performance 8 Consolidated Balance Sheet 9 Consolidated Statements of Cash Flows 10 Research & Development and Capital Investment 11 Risk Factors 12 Outlook and Guidance (FY2024) 13 Notes and Glossary 14 \n","pages":[{"image_uri":"","page_id":0},{"image_uri":"","page_id":1}]},{"chunk_id":"18c0eb6d4d8e4e95b35d8ff78966de40_1","chunk_position":1,"chunk_to_embed":"Company: Meridian Motor Corporation\nDocument Type: Annual Report\nFiscal Year: FY2023\nStock Code: 7000\nStock Exchange: Tokyo Stock Exchange (Prime Market)\nDocument Title: Financial Highlights — Five-Year Summary\nSection: Letter from the President and CEO\nTables: over the past five fiscal years. In FY2023, net revenue was ¥16.91 trillion and net income attrib... (¥ billions (except per-share), FY2019, FY2020, FY2021, FY2022, ...)\nContains: table, text\n\nMeridian Motor Corporation created this annual report for fiscal year ended March 31, 2023.\n\nTable summary: Fiscal 2019‑2023 financial and operational metrics for Meridian Motor Corporation, including revenue, profit, EPS, and sales.\n\nContent:\n\nAll figures in this report are fictional and provided solely for CS4603 coursework.\n\nTo our shareholders, customers, and employees: Meridian Motor Corporation delivered record results in the fiscal year ended March 31, 2023. Consolidated net revenue rose 16.2% to ¥16,910 billion (¥16.91 trillion), and operating profit increased 24.2% to ¥1,124 billion. Net income attributable to owners of the parent reached ¥1,107 billion, our highest ever.\n\nDemand recovered strongly across all major markets as supply-chain constraints eased. Global vehicle unit sales grew to 4.07 million units, up from 3.68 million a year earlier, led by our electrified line-up in North America and Asia. Our Motorcycle business remained the profitability anchor of the group, with a 14.3% operating margin.\n\nWe continued to invest for the long term. Research and development expense rose to ¥880 billion, or 5.2% of net revenue, concentrated on battery-electric platforms, software-defined vehicle architecture, and advanced driver assistance. Capital expenditure was ¥590 billion.\n\nLooking ahead to FY2024, we forecast net revenue o

In [0]:
from databricks.ai_search.client import VectorSearchClient

client = VectorSearchClient(
    disable_notice=True
)

print("Client initialized.")

Client initialized.


In [0]:
try:
    endpoint_details = client.get_endpoint(
        VECTOR_SEARCH_ENDPOINT
    )

    print(
        "Endpoint already exists:",
        VECTOR_SEARCH_ENDPOINT,
    )

except Exception as exc:
    message = str(exc).upper()

    if (
        "NOT_FOUND" not in message
        and "DOES NOT EXIST" not in message
        and "RESOURCE_DOES_NOT_EXIST" not in message
    ):
        raise

    print(
        "Creating endpoint:",
        VECTOR_SEARCH_ENDPOINT,
    )

    client.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT,
        endpoint_type="STANDARD",
    )

Endpoint already exists: pa4-vs-endpoint


In [0]:
import time
import pprint

endpoint_ready = False

for attempt in range(60):
    endpoint_details = client.get_endpoint(
        VECTOR_SEARCH_ENDPOINT
    )

    endpoint_status = endpoint_details.get(
        "endpoint_status",
        endpoint_details.get("status", {}),
    )

    print(
        f"{attempt + 1:02d}:",
        endpoint_status,
    )

    status_text = str(endpoint_status).upper()

    if (
        "ONLINE" in status_text
        or "READY" in status_text
    ):
        endpoint_ready = True
        print("Endpoint is ready.")
        break

    if "FAIL" in status_text:
        raise RuntimeError(
            f"Endpoint provisioning failed: "
            f"{endpoint_details}"
        )

    time.sleep(30)

if not endpoint_ready:
    raise TimeoutError(
        "Vector Search endpoint did not become ready within 30 minutes."
    )

01: {'state': 'ONLINE'}
Endpoint is ready.


In [0]:
existing_index = None
existing_details = None

try:
    existing_index = client.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=VECTOR_SEARCH_INDEX,
    )

    existing_details = existing_index.describe()

    print("Existing index found.")
    pprint.pp(existing_details)

except Exception as exc:
    message = str(exc).upper()

    if (
        "NOT_FOUND" not in message
        and "DOES NOT EXIST" not in message
        and "RESOURCE_DOES_NOT_EXIST" not in message
    ):
        raise

    print("No existing index was found.")

Existing index found.
{'name': 'cs4603.pa4.pa4_analyst_index',
 'endpoint_name': 'pa4-vs-endpoint',
 'primary_key': 'chunk_id',
 'index_type': 'DELTA_SYNC',
 'delta_sync_index_spec': {'source_table': 'cs4603.pa4.pa4_analyst_chunks',
                           'embedding_source_columns': [{'name': 'chunk_to_retrieve',
                                                         'embedding_model_endpoint_name': 'databricks-gte-large-en'}],
                           'pipeline_type': 'TRIGGERED',
                           'pipeline_id': 'ffc4469f-8844-46d7-a630-1fa89a29c942'},
 'status': {'detailed_state': 'ONLINE_NO_PENDING_UPDATE',
            'message': 'Index creation succeeded. Check latest status: '
                       'https://dbc-01190470-5ed4.cloud.databricks.com/explore/data/cs4603/pa4/pa4_analyst_index',
            'indexed_row_count': 2,
            'triggered_update_status': {'last_processed_commit_version': 0,
                                        'last_processed_commit_t

In [0]:
if existing_details is not None:
    embedding_columns = (
        existing_details
        .get("delta_sync_index_spec", {})
        .get("embedding_source_columns", [])
    )

    current_embedding_source = (
        embedding_columns[0].get("name")
        if embedding_columns
        else None
    )

    print(
        "Current embedding source:",
        current_embedding_source,
    )

    assert current_embedding_source == "chunk_to_embed", (
        f"Existing index embeds {current_embedding_source!r}; expected "
        "'chunk_to_embed'. Delete that index from Catalog Explorer once, "
        "then rerun from this cell."
    )

print("Index configuration is compatible.")

Current embedding source: chunk_to_retrieve
Deleting incorrectly configured index: cs4603.pa4.pa4_analyst_index
Delete request submitted.


In [0]:
# Intentionally do not delete/recreate an index during Run All.
# Immediate recreation with the same name can leave stale pipeline metadata.
print("No index deletion is required.")

01: Waiting for index deletion...
Old index has been deleted.


In [0]:
# Create a new index when needed, otherwise synchronize the existing one.
if existing_index is None:
    print("Creating Delta Sync index:", VECTOR_SEARCH_INDEX)
    index = client.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=VECTOR_SEARCH_INDEX,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_to_embed",
        embedding_model_endpoint_name=EMBEDDINGS_ENDPOINT,
    )
    print("Index creation request submitted.")
else:
    index = existing_index
    print("Synchronizing existing index:", VECTOR_SEARCH_INDEX)
    index.sync()
    print("Index synchronization request submitted.")

index_ready = False

for attempt in range(80):
    index = client.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=VECTOR_SEARCH_INDEX,
    )

    details = index.describe()
    status = details.get("status", {})

    state = status.get(
        "detailed_state",
        "UNKNOWN",
    )
    message = status.get("message", "")
    ready = status.get("ready", False)

    print(
        f"{attempt + 1:02d}: "
        f"{state} — {message}"
    )

    sync_complete = (
        ready
        and "TRIGGERED_UPDATE" not in str(state).upper()
        and "PROVISION" not in str(state).upper()
    )

    if sync_complete:
        index_ready = True
        print(
            "Index is ready:",
            VECTOR_SEARCH_INDEX,
        )
        break

    if "FAIL" in str(state).upper():
        raise RuntimeError(
            f"Index creation failed: {status}"
        )

    time.sleep(30)

if not index_ready:
    raise TimeoutError(
        "Vector Search index did not finish synchronizing within 40 minutes."
    )

Creating Delta Sync index: cs4603.pa4.pa4_analyst_index
Index creation request submitted.
01: PROVISIONING_INDEX — Delta sync Index creation is pending. Check latest status: https://dbc-01190470-5ed4.cloud.databricks.com/explore/data/cs4603/pa4/pa4_analyst_index
02: ONLINE_TRIGGERED_UPDATE — Index is online but is currently is in the process of re-syncing initial data. Check latest status: https://dbc-01190470-5ed4.cloud.databricks.com/explore/data/cs4603/pa4/pa4_analyst_index
Index is ready: cs4603.pa4.pa4_analyst_index


In [0]:
details = index.describe()

embedding_columns = (
    details
    .get("delta_sync_index_spec", {})
    .get("embedding_source_columns", [])
)

assert embedding_columns, (
    "The index does not report an embedding source."
)

assert (
    embedding_columns[0]["name"]
    == "chunk_to_embed"
), embedding_columns

assert details["status"]["ready"] is True, (
    details["status"]
)

pprint.pp(
    {
        "name": details.get("name"),
        "endpoint": details.get("endpoint_name"),
        "embedding_source": (
            embedding_columns[0]["name"]
        ),
        "embedding_model": (
            embedding_columns[0].get(
                "embedding_model_endpoint_name"
            )
        ),
        "status": details.get("status"),
    }
)

{'name': 'cs4603.pa4.pa4_analyst_index',
 'endpoint': 'pa4-vs-endpoint',
 'embedding_source': 'chunk_to_embed',
 'embedding_model': 'databricks-gte-large-en',
 'status': {'detailed_state': 'ONLINE_TRIGGERED_UPDATE',
            'message': 'Index is online but is currently is in the process of '
                       're-syncing initial data. Check latest status: '
                       'https://dbc-01190470-5ed4.cloud.databricks.com/explore/data/cs4603/pa4/pa4_analyst_index',
            'indexed_row_count': 7,
            'triggered_update_status': {'last_processed_commit_version': 8,
                                        'last_processed_commit_timestamp': '2026-07-12T12:34:02Z',
                                        'triggered_update_progress': {'latest_version_currently_processing': 8,
                                                                      'num_synced_rows': 7,
                                                                      'total_rows_to_sync': 7,
       

In [0]:
results = index.similarity_search(
    query_text=(
        "What was Meridian Motor Corporation's "
        "revenue in 2023?"
    ),
    columns=[
        "chunk_id",
        "chunk_to_embed",
        "chunk_to_retrieve",
        "source",
        "page",
    ],
    num_results=4,
)

pprint.pp(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'manifest': {'column_count': 6,
              'columns': [{'name': 'chunk_id'},
                          {'name': 'chunk_to_embed'},
                          {'name': 'chunk_to_retrieve'},
                          {'name': 'source'},
                          {'name': 'page'},
                          {'name': 'score'}]},
 'result': {'row_count': 4,
            'data_array': [['4db559e9211140c8b8d4aa2e947bbba3_3',
                            'Code: 7000\n'
                            'Company: Meridian Motor Corporation\n'
                            'Document Type: Annual Report\n'
                            'Fiscal Year: FY2023\n'
                            'Stock Exchange: Tokyo Stock Exchange (Prime '
                            'Market)\n'
                            

In [0]:
print(
    f"""
Add these values to your local .env file:

UC_CATALOG={CATALOG}
UC_SCHEMA={SCHEMA}
SOURCE_TABLE={SOURCE_TABLE}
VECTOR_SEARCH_ENDPOINT={VECTOR_SEARCH_ENDPOINT}
VECTOR_SEARCH_INDEX={VECTOR_SEARCH_INDEX}
EMBEDDINGS_ENDPOINT={EMBEDDINGS_ENDPOINT}
""".strip()
)

Add these values to your local .env file:

UC_CATALOG=cs4603
UC_SCHEMA=pa4
SOURCE_TABLE=cs4603.pa4.pa4_analyst_chunks
VECTOR_SEARCH_ENDPOINT=pa4-vs-endpoint
VECTOR_SEARCH_INDEX=cs4603.pa4.pa4_analyst_index
EMBEDDINGS_ENDPOINT=databricks-gte-large-en


In [0]:
# Standalone LangChain verification; no local project import is required.
from databricks_langchain import DatabricksVectorSearch

vector_store = DatabricksVectorSearch(
    index_name=VECTOR_SEARCH_INDEX,
    columns=["chunk_id", "chunk_to_retrieve", "source", "page"],
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

docs = retriever.invoke(
    "What was Meridian Motor Corporation's revenue in 2023?"
)

assert docs, "LangChain retrieval returned no documents."

for doc in docs:
    print(doc.metadata.get("chunk_to_retrieve", doc.page_content)[:500])
    print(doc.metadata)
    print("-" * 80)

/home/spark-af301361-f558-432f-8404-80/.ipykernel/308/command-8282376280077274-1379579681:4: DeprecationWarning: The `endpoint` parameter is deprecated and will be ignored. The endpoint is automatically inferred from the index name.
  vector_store = DatabricksVectorSearch(


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


---------------------------------------------------------------------------
NotFound                                  Traceback (most recent call last)
File <command-8282376280077274>, line 11
      4 vector_store = DatabricksVectorSearch(
      5     endpoint=VECTOR_SEARCH_ENDPOINT,
      6     index_name=VECTOR_SEARCH_INDEX,
      7     columns=["chunk_id", "chunk_to_retrieve", "source", "page"],
      8 )
      9 retriever = vector_store.as_retriever(search_kwargs={"k": 4})
---> 11 docs = retriever.invoke(
     12     "What was Meridian Motor Corporation's revenue in 2023?"
     13 )
     15 for doc in docs:
     16     print(doc.page_content[:500])

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-af301361-f558-432f-8404-80e853e8c8a2/lib/python3.12/site-packages/langchain_core/retrievers.py:222, in BaseRetriever.invoke(self, input, config, **kwargs)
    220 kwargs_ = kwargs if self._expects_other_args else {}
    221 if self._new_arg_supported:
--> 222     result = self._get_relevan